# Giai đoạn 2 — Tiền xử lý dữ liệu

Đọc 14 CSV gốc, kiểm tra khóa, liên kết và kiểu dữ liệu, sau đó tạo hai bảng phân tích ở cấp đơn hàng và cấp dòng hàng. Nếu bản CSV đã có, notebook so từng dòng với kết quả tính lại.

Giai đoạn 2 bảo toàn 14 nguồn và tạo hai view dùng chung. Giữ bất thường đã được duyệt, phân biệt nguồn, diễn giải và điều chưa xác minh. Notebook thực thi code trong `src/preprocessing`, không chứa bản sao thuật toán nguồn. Không huấn luyện mô hình, tính RFM, scaling hoặc lag/rolling.

Kết quả bên dưới được tính thực tế trong kernel sạch. Tiền kiểm chứng dùng Decimal; biểu đồ có thể chuyển float chỉ để hiển thị.

In [32]:
from pathlib import Path
import os, sys
sys.dont_write_bytecode=True

import os

import sys

start = Path(os.environ.get("RETAIL_ANALYTICS_ROOT", Path.cwd())).resolve()
ROOT = next((path for path in (start, *start.parents)
             if (path / "notebook/ham_dung_chung.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Mở thư mục gốc của dự án hoặc đặt RETAIL_ANALYTICS_ROOT.")
sys.path.insert(0, str(ROOT / "notebook"))

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display,Markdown
from ham_dung_chung import preprocessing
CommonPreprocessing = preprocessing().CommonPreprocessing
ORDERS_SCHEMA = preprocessing().ORDERS_SCHEMA
ITEMS_SCHEMA = preprocessing().ITEMS_SCHEMA
from ham_dung_chung import delivery
validate_orders = delivery().validate_orders
validate_items = delivery().validate_items
validate_financials = delivery().validate_financials
validate_supporting = delivery().validate_supporting
publish = delivery().publish
from ham_dung_chung import foundation
exact_sum = foundation().exact_sum
%matplotlib inline
pd.set_option('display.max_rows',16)
pd.set_option('display.max_columns',9)
pd.set_option('display.max_colwidth',70)
plt.rcParams.update({'figure.figsize':(10,4),'font.family':'DejaVu Sans','axes.spines.top':False,'axes.spines.right':False})
print('PROJECT_ROOT:',ROOT)
print('Python:',sys.version.split()[0],'Pandas:',pd.__version__)
from ham_dung_chung import accepted_processed_outputs

# Dùng thư mục khác khi cần tạo một bộ kết quả mới.
OUTPUT_DIR = ROOT / "data/processed"
existing_outputs = accepted_processed_outputs(OUTPUT_DIR, ORDERS_SCHEMA, ITEMS_SCHEMA)
run = None if existing_outputs else CommonPreprocessing(ROOT, chunk_size=25000)
if existing_outputs:
    print("Đã có đủ hai CSV đúng cấu trúc cột. Dùng lại đầu ra.")

PROJECT_ROOT: E:\SGU\Ky1_4\DA\DA_Datathon2026
Python: 3.11.7 Pandas: 2.2.3
Đã có đủ hai CSV đúng cấu trúc cột. Dùng lại đầu ra.


## 1. Tổng quan và đọc 14 bảng

Đọc theo lô, tách nguyên văn và kiểu làm việc, kiểm tra header, khóa xuyên lô và FK trước JOIN. SQLite chỉ là vùng làm việc tạm; tiền lưu chuỗi Decimal. Hash được đối chiếu trước khi công bố. Không xuất bản sao 14 nguồn.

In [33]:
if run is not None:
    run.load(progress=lambda name,n:print(f'{name}: {n:,} bản ghi'))
    overview=pd.DataFrame([{'Bảng':n,'Dòng nguồn':s['rows'],'Cột nguồn':len(s['columns']),'SHA-256':s['sha256']} for n,s in run.sources.items()])
    display(overview)

## 2. Hai mươi bốn chính sách đã duyệt

G = giữ nguyên; C = cờ; K = kiểm chứng; V = view; B = parse/tính trường dẫn xuất. Các số đếm trong bảng mô tả dữ liệu quan sát của dự án.

| DQ | Chính sách | Kiểm chứng | Phạm vi |
|---|---|---|---|
| 01 | Giữ 477.453 đơn trước đăng ký; không sửa ngày/loại đơn | V01.parse_date/analyze; G+C+K; cờ order_before_signup_flag tính trực tiếp, thiếu căn cứ là NA | Parse nền ở 2.3.1.3; cờ đủ từng đơn ở 2.3.2 |
| 02 | Parse customers.signup_date theo `%Y-%m-%d` (YYYY-MM-DD); giữ định dạng các cột ngày khác và ngày thực tế | V01.parse_date, V06.strict_dates; G+B+K; kiểm tra cú pháp/lịch/ngày nhuận, raw bất biến | Nền tảng |
| 03 | Parser signup khớp CSV hiện tại: YYYY-MM-DD | Dùng V01 với `%Y-%m-%d`, regex `\d{4}-\d{2}-\d{2}`; không sửa nguồn | Nhận định parser signup ISO sai chỉ áp dụng cho bản raw cũ |
| 04 | Giữ NULL và hai vị trí mã trên dòng hàng | V06.null_kind; G+K; raw/states giữ dấu vết, không điền mã | Nền tảng |
| 05 | Giữ cả sáu trạng thái trong dữ liệu chung | V02.analyze ở bước view; G+K; không lọc khi đọc, bảo toàn tập trạng thái | Đọc nền; view đơn ở 2.3.2; lọc riêng theo bài toán |
| 06 | Phân biệt Gross, Discount, Net, Payment, sales.Revenue; Revenue đối chứng Gross trước giảm trên mọi trạng thái; Revenue−COGS chưa là lợi nhuận thực | V02.analyze/compare/independent_audit; G+B+V+K; Discount một lần/dòng, tổng hợp con trước JOIN | Decimal nền; đối soát/view ở 2.3.2/2.3.3 |
| 07 | Giữ products.cogs và sales.COGS, không tạo giá vốn lịch sử/ép bằng nhau | V02.decimal_value/compare; G+B+K; COGS nguồn và đối chứng riêng, không làm tròn cent | Decimal nền; đối chứng sau |
| 08 | Giữ cả 714.669 items, gồm cặp lặp; ID ổn định truy nguyên | G+B+K; source_row_id trước sort/JOIN; mỗi bản ghi nguồn một ID | ID nền; order_items_common ở 2.3.3 |
| 09 | Giữ 39.939 returns/return_id, kể cả cặp lặp | V03.analyze/independent_audit; G+V+K; tổng hợp đúng cấp trước nối | Khóa/FK nền; tổng hợp sau |
| 10 | Giữ đơn thiếu shipment, LEFT JOIN, gắn cờ; không tạo ngày | NV03.build_shipping_views/build_cross034; G+C+V; một dòng đơn sau nối | Nền chỉ đọc; orders_common ở 2.3.2 |
| 11 | Status returned và có returns là hai thông tin riêng | V03.analyze; G+C+K; bảng chéo đủ bốn nhóm, không suy nhóm thiếu từ chênh tổng | Khóa/FK nền; bảng chéo/cờ ở 2.3.2 |
| 12 | Giữ refund vượt payment, không cap/xóa/sửa | V03.analyze/financial_difference; G+C+K; tổng hợp riêng từng bảng theo đơn, hai phép so độc lập | Decimal nền; đối soát ở 2.3.2 |
| 13 | Giữ inventory.units_sold khác tổng quantity | V04.analyze/compare_measure; G+V+K; không sửa hai lượng để ép khớp | Kiểu/FK nền; đối soát ở bước sau/Bài toán 1 |
| 14 | Không fill 0/ffill/bfill hoặc tạo snapshot cho cặp thiếu | V04/NV05 coverage; G+K; báo giao/hợp và ngoài phạm vi, không sinh tồn kho | Nền giữ mọi dòng; báo coverage sau |
| 15 | Residual chỉ giữa tháng liên tiếp, residual=0 chưa chứng minh kiểm kê, không cộng hàng trả vào received | NV05.adjacent54; G+B+K; khoảng tháng thiếu không bắc cầu | Bước kiểm chứng sau/Bài toán 1 |
| 16 | Giữ reorder_flag=0 ở 60.247 dòng; biến hằng, không nhãn giả | V05.analyze/assess; G+K; mốc theo snapshot, không cấm giá trị khác ở mọi dataset | Nền parse giữ giá trị; hồ sơ biến sau |
| 17 | Giữ hai mã; Discount một lần/dòng; lượt mã không phân bổ tiền | V06.analyze, logic NV06; G+V+K; view slot khóa (item_row_id, slot) tách khỏi view tiền | Giữ slot nền; view nếu cần sau 2.3.3 |
| 18 | Giữ chương trình/mã/Discount; chỉ ứng viên dưới ngưỡng Gross/Net | NV06 cell code 43; C+K; so tổng đơn, không gọi vi phạm/loại mã | Chưa xây ở nền; bước phân tích/view sau |
| 19 | Giữ bounce theo thang nguồn; ba loại kênh khác cấp, không attribution bằng tên | V07/V08.analyze; G+K; không đổi thang hoặc tạo FK kênh | Nền giữ kiểu/nhãn; kiểm chứng/chỉ tiêu sau |
| 20 | Giữ khoảng quan sát từng bảng; phân biệt thiếu/trong/ngoài phủ, không suy sau cuối kỳ | V04/V07/V08/V09; G+C+K; không sinh dòng/ngày/0 | Nền không fill; báo coverage ở bước sau |
| 21 | RFM delivered 04/07/2012–31/12/2022, chốt 31/12/2022; R từ đơn delivered gần nhất, F distinct order_id, M tổng quantity×unit_price−discount; không cộng Payment/trừ Refund; giữ 36.815 khách ngoài tập với R không xác định | NV02.build_behavior_scope/build_value_orders; V+B+K; chưa chuẩn hóa/phân cụm | Bài toán 2; nền không tính RFM |
| 22 | Giữ Date/Revenue/COGS của 3.833 ngày Sales; 548 ngày template chỉ lịch/schema, không nhãn; chưa lag/rolling/scaling/model | V09.analyze/inspect_roundtrip; G+V+K; không trộn template vào lịch sử | Nền đọc hai bảng riêng; chuẩn bị mô hình thuộc Bài toán 3 |
| 23 | Giữ baseline gốc; không kế thừa lỗi TEST_FILE/đường dẫn/đánh giá leakage | G+K; hash baseline, không thực thi từ nền | Backtest Bài toán 3 |
| 24 | Không sửa nguồn để test đạt; kiểm tra phụ thuộc, đường dẫn và tính toàn vẹn, công bố sau kiểm chứng | digest, V02.require_key/require_link, notebook_integrity; K; nguồn và file nghiệm thu bất biến | Root/hash/ID nền; hoàn thiện tái lập và công bố ở GD2 |

## 3. Xây dựng orders_common

Orders là tập đơn gốc, giữ nguyên tám cột và mọi trạng thái. Items nối Products theo many-to-one để lấy COGS tham chiếu; Items, Payments, Returns, Shipments được tổng hợp độc lập rồi LEFT JOIN một-một. Không nhân tiền theo số bản ghi con.

Không có Returns khác Refund bằng 0: tiền và cờ so sánh không đủ căn cứ được giữ NA. DQ-11 dùng bảng chéo đầy đủ, không suy từ chênh hai tổng.

In [34]:
if run is not None:
    orders=run.build_orders_common()
    orders_report=validate_orders(run)
    display(orders.head(6)[['order_id','order_status','item_rows','gross_amount','net_amount','payment_amount','refund_amount_observed','cogs_check_amount']])
    display(pd.DataFrame(orders_report['dq11_cross']))
    display(pd.DataFrame(orders_report['flags']).T.rename(columns={'true':'True','false':'False','unknown':'NA'}))
    print('Schema:',len(orders.schema),'cột; số đơn:',len(orders),'đủ điều kiện:',orders_report['accepted'])

## 4. JOIN, trạng thái và những bất thường được giữ

Các nhóm cờ có thể giao nhau, không cộng số đếm thành tổng lỗi. Returned thiếu Returns, delivered thiếu shipment, items lặp cặp và refund vượt payment đều giữ nguyên.

In [35]:
if run is not None:
    status=pd.Series(orders_report['status_distribution']).sort_values(ascending=False)
    flags=pd.Series({k:v['true'] for k,v in orders_report['flags'].items()})
    fig,axes=plt.subplots(1,2,figsize=(14,4))
    status.plot.bar(ax=axes[0],color='#307c9b');axes[0].set(title='Tất cả trạng thái được giữ',ylabel='Đơn',xlabel='Trạng thái')
    flags.index=['Trước đăng ký','Items lặp cặp','Nhiều Returns','Delivered thiếu ship','Returned thiếu Returns','Refund > Payment','Refund vượt 0,05']
    flags.plot.barh(ax=axes[1],color='#c17b35');axes[1].set(title='Cờ theo đơn — nhóm có thể giao nhau',xlabel='Đơn')
    plt.tight_layout();plt.show()
    display(pd.DataFrame([c for c in run.checks if c['check'].startswith(('FK:','orders:'))])[['check','status']])

## 5. Xây dựng order_items_common và bảo toàn tiền

Một dòng cho mỗi bản ghi nguồn, kể cả trùng cặp. Giữ hai slot promo; Discount chỉ tính một lần. COGS là tham chiếu từ Products. Payment/Refund cấp đơn không được chép thành chỉ tiêu cộng trên dòng.

In [36]:
if run is not None:
    items=run.build_order_items_common()
    items_report=validate_items(run)
    financial=validate_financials(run)
    display(items.head(6)[['item_row_id','order_id','quantity','promo_id','promo_id_2','line_gross','line_discount','line_net','line_cogs_check']])
    display(pd.DataFrame([{'Chỉ tiêu':k,'Tổng nguồn độc lập (Decimal)':str(v)} for k,v in financial['source_totals'].items()]))
    print('Dòng hàng:',len(items),'cột:',len(items.schema),'đủ điều kiện:',items_report['accepted'])
    print('Oracle: CSV/Decimal trực tiếp; không dùng COGS dữ liệu notebook đã qua float làm chuẩn tuyệt đối.')

## 6. Kiểm chứng nguồn bổ trợ

V09 được tính trực tiếp từ CSV doanh thu và tệp mẫu. GD2 kiểm tra riêng phạm vi khách hàng cho RFM và các tháng tồn kho đủ dữ liệu. Inventory không lấp snapshot hay ép units_sold khớp giao dịch; residual chỉ xét tháng liền nhau và chưa xác nhận kiểm kê thực tế.

RFM chỉ xác nhận phạm vi; chưa tính R/F/M. Dữ liệu dự báo bàn giao lịch sử Sales và lịch tương lai, không coi giá trị template là nhãn.

In [37]:
if run is not None:
    support=validate_supporting(run)
    display(pd.DataFrame([support['rfm_scope']]))
    display(pd.DataFrame([support['inventory_residual']]))
    display(pd.DataFrame([{'Nguồn lịch sử':support['forecast_handoff']['history_source'],'Ngày tương lai':len(support['forecast_handoff']['calendar']),'Bắt đầu':support['forecast_handoff']['calendar'][0],'Kết thúc':support['forecast_handoff']['calendar'][-1],'Giá trị mẫu là nhãn':support['forecast_handoff']['template_values_are_labels']}]))

## 7. Minh họa dữ liệu bàn giao dự báo

Hai chỉ tiêu lịch sử giữ nguyên nguồn. Chuyển float dưới đây chỉ phục vụ biểu đồ; đối soát/xuất dữ liệu vẫn dùng Decimal. Không xây đặc trưng hoặc chia tập chung cho ba bài toán.

In [38]:
if run is not None:
    from decimal import Decimal
    sales=pd.read_sql_query('SELECT _p_Date AS Date,_p_Revenue AS Revenue,_p_COGS AS COGS FROM sales ORDER BY _p_Date',run.connection)
    sales['Date']=pd.to_datetime(sales.Date)
    for col in ['Revenue','COGS']:sales[col]=sales[col].map(Decimal)
    monthly=sales.groupby(sales.Date.dt.to_period('M'))[['Revenue','COGS']].agg(lambda x:exact_sum(x))
    fig,ax=plt.subplots(figsize=(12,4))
    for col in ['Revenue','COGS']:ax.plot(monthly.index.to_timestamp(),monthly[col].map(float)/1000000,label=col)
    ax.set(title='Sales: hai chỉ tiêu được giữ đầy đủ',ylabel='Triệu đơn vị tiền nguồn',xlabel='Tháng');ax.legend();plt.tight_layout();plt.show()

## 8. Tổng hợp trước/sau và truy nguyên

Nguồn không bị thay đổi. Hai view là lớp bổ sung; số đơn/dòng và mọi khoản tiền phải bảo toàn. Cùng file/hash/thứ tự bản ghi CSV cho cùng ID nguồn; xuống dòng trong ô không tạo thêm bản ghi.

In [39]:
if run is not None:
    display(pd.DataFrame([{'Đầu vào':n,'Dòng nguồn':s['rows'],'Xử lý':'Giữ nguyên nguồn; kiểm tra schema/kiểu/khóa/FK'} for n,s in run.sources.items()]))
    display(pd.DataFrame([{'View':'orders_common','Dòng đầu vào':run.sources['orders']['rows'],'Dòng đầu ra':len(orders),'Cột':len(orders.schema)}, {'View':'order_items_common','Dòng đầu vào':run.sources['order_items']['rows'],'Dòng đầu ra':len(items),'Cột':len(items.schema)}]))
    example=orders.head(1).iloc[0].order_id
    display(pd.DataFrame(run.lineage_for_order(example)))


## 9. Công bố sau nghiệm thu và bàn giao

Chỉ công bố nếu từng view đạt, có oracle độc lập và kiểm chứng bổ trợ. CSV dùng token `\N` cho NA; đọc lại bằng `read_processed()` với schema manifest, không để pandas suy ID/ZIP hoặc float. Round-trip so chính xác từng dòng với view. File đã tồn tại chỉ được xác minh nếu cùng manifest; không ghi đè âm thầm.

In [40]:
if run is None:
    display(pd.DataFrame(existing_outputs).T)
else:
    delivery=publish(run, output_dir=OUTPUT_DIR)
    print('Trạng thái:',delivery['status'])
    display(pd.DataFrame([{'View':n,'Dòng round-trip':e['rows'],'So từng dòng chính xác':e['exact_row_comparison'],'SHA-256 CSV':e['sha256']} for n,e in delivery['outputs'].items()]))
    display(pd.DataFrame([{'Phép kiểm chứng': c['check'], 'Trạng thái': c['status']} for c in run.checks]))

,Tệp,Trạng thái
orders_common,E:\SGU\Ky1_4\DA\DA_Datathon2026\data\processed\orders_common.csv,Dùng lại
order_items_common,E:\SGU\Ky1_4\DA\DA_Datathon2026\data\processed\order_items_common.csv,Dùng lại


## 10. Kết luận và giới hạn

- Bài toán 1: orders_common và order_items_common; mọi trạng thái và bất thường vẫn giữ, cờ không phải kết luận vi phạm nghiệp vụ.
- Bài toán 2: lọc delivered theo hợp đồng riêng, chốt 31/12/2022; giữ khách ngoài phạm vi và lý do. Chưa scaling/clustering.
- Bài toán 3: dùng Sales Date/Revenue/COGS, lịch và cột đích từ tệp mẫu. Không lấy giá trị mẫu làm nhãn. Status/Refund/Shipment trong view là snapshot cuối kỳ, không mặc định sẵn có tại mỗi fold lịch sử.
- Chưa xác minh tiền tệ/thuế/phí, giá vốn lịch sử, nghĩa NULL danh mục, stacking/eligibility, cấp kho và attribution marketing. Không suy lợi nhuận thực tế từ Revenue−COGS.
- Không sửa raw, baseline hoặc notebook lịch sử. Các kiểm thử chạy và trạng thái Git được lưu/báo riêng; không tự commit/push.

In [41]:
if run is not None:
    run.check_sources()
    print('Số kiểm tra thực tế:',len(run.checks))
    print('Không có kiểm tra unsafe:',all(c['status']=='passed' for c in run.checks))
    print('Không có RFM/model/scaling/lag/rolling được thực hiện.')
    run.close()